In [ ]:
!pip install pandas requests yfinance --quiet

import pandas as pd
import requests
import yfinance as yf
import warnings
import time
import numpy as np
from datetime import datetime, timedelta

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. CONFIGURATION
# ==========================================
# PASTE YOUR TWELVE DATA API KEY HERE
TD_API_KEY    = "7c5a55559d164d4ebca0d23a5baea839" 

# SETTINGS
LOOKBACK_DAYS   = 60      
DISCOUNT_LEVEL  = 0.20   
PREMIUM_LEVEL   = 0.80    
TARGET_PROCESSED = 80    # We want to scan 80 SAFE stocks
EARNINGS_BUFFER = 21     # Days
# --- EDITED FILTERS ---
MIN_PRICE       = 25     # Only scan stocks > $25
MAX_PRICE       = 500    # Only scan stocks < $500

# TRIGGER SETTINGS (Reversion)
TRIGGER_EMA = 8      
RSI_PERIOD  = 14     
RSI_BUY_MAX = 45     
RSI_SELL_MIN = 55    
FAST_EMA    = 5
SLOW_EMA    = 20

# Watchlist
RAW_TICKERS = [
    "A", "AA", "AAL", "AAP", "AAPL", "ABBV", "ABNB", "ABT", "ACN", "ADBE", "ADI", "ADM", "ADP", "ADSK", "AEP", "AES", "AFL", "AFRM", "AGNC", "AI", "AIG", "AKAM", "ALB", "ALK", "ALL", "AMAT", "AMD", "AMGN", "AMRN", "AMZN", "APA", "APH", "APTV", "AVGO", "AXP", "BA", "BABA", "BAC", "BAX", "BBY", "BEN", "BIDU", "BIIB", "BITO", "BK", "BKR", "BMY", "BP", "BSX", "BX", "BYND", "C", "CAH", "CAT", "CB", "CCI", "CCJ", "CCL", "CF", "CFG", "CHWY", "CI", "CL", "CLF", "CLX", "CMCSA", "CME", "CNC", "CNP", "COF", "COIN", "COP", "COST", "CPB", "CPRI", "CRM", "CRON", "CRWD", "CSCO", "CSX", "CTVA", "CVNA", "CVS", "CVX", "CZR", "D", "DAL", "DD", "DE", "DELL", "DHI", "DHR", "DIA", "DIS", "DKNG", "DLR", "DOCU", "DOW", "DRI", "DVN", "DXC", "EA", "EBAY", "ED", "EEM", "EFA", "EIX", "EL", "EMR", "ENPH", "EOG", "EQR", "EQT", "ETSY", "EVRG", "EW", "EWJ", "EWW", "EWY", "EWZ", "EXC", "EXPE", "F", "FANG", "FAST", "FCX", "FDX", "FE", "FI", "FITB", "FOXA", "FSLR", "FTI", "FTV", "FXE", "FXI", "GD", "GDX", "GE", "GILD", "GLD", "GLW", "GM", "GME", "GOOG", "GOOGL", "GPRO", "GPS", "GS", "HAL", "HBAN", "HBI", "HCA", "HD", "HIG", "HLT", "HOG", "HOLX", "HON", "HPE", "HPQ", "HYG", "IBB", "IBIT", "IBM", "ICE", "IEF", "INTC", "IP", "IPG", "IRM", "IVZ", "IWM", "IYR", "JCI", "JD", "JETS", "JNJ", "JNPR", "JPM", "K", "KHC", "KMI", "KO", "KR", "KRE", "KSS", "KWEB", "LEN", "LI", "LKQ", "LLY", "LNC", "LOW", "LQD", "LUMN", "LUV", "LVS", "LYB", "LYFT", "MA", "MAR", "MARA", "MAS", "MCD", "MCHP", "MDLZ", "MDT", "MET", "META", "MGM", "MMM", "MO", "MOS", "MPC", "MRK", "MRNA", "MRVL", "MS", "MSFT", "MSTR", "MTB", "MTCH", "MU", "NCLH", "NEE", "NEM", "NET", "NFLX", "NKE", "NOW", "NRG", "NTAP", "NTES", "NVAX", "NVDA", "NWL", "NWSA", "O", "OIH", "OKE", "OMC", "ON", "ORCL", "OXY", "PARA", "PBR", "PDD", "PENN", "PEP", "PFE", "PFG", "PG", "PGR", "PINS", "PLTR", "PNC", "PPL", "PRU", "PSX", "PTON", "PYPL", "QCOM", "QQQ", "RBLX", "RCL", "RF", "RIG", "RIOT", "RIVN", "ROKU", "ROST", "RTX", "RUN", "SBUX", "SCHD", "SCHW", "SEDG", "SHOP", "SIRI", "SLB", "SLV", "SMCI", "SMH", "SNAP", "SNOW", "SO", "SOFI", "SOXL", "SOXS", "SPG", "SPX", "SPXL", "SPXS", "SPY", "SQQQ", "STX", "SWKS", "SYF", "SYY", "T", "TAP", "TBT", "TCOM", "TDOC", "TFC", "TGT", "TJX", "TLT", "TMO", "TMUS", "TPR", "TQQQ", "TRIP", "TRV", "TSLA", "TSM", "TSN", "TT", "TTD", "TTWO", "TXN", "U", "UA", "UAA", "UAL", "UBER", "ULTA", "UNG", "UNH", "UNP", "UPS", "UPST", "URBN", "USB", "USO", "UVXY", "V", "VALE", "VFC", "VGK", "VTR", "VXX", "VZ", "WAB", "WBA", "WBD", "WDC", "WFC", "WM", "WMB", "WMT", "WU", "WY", "WYNN", "X", "XBI", "XEL", "XHB", "XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY", "XOM", "XOP", "XRT", "XRX", "XSP", "YELP", "YINN", "ZION", "ZM"
]
tickers = list(set(RAW_TICKERS))

# YAHOO FINANCE FORMATTING MAP
TICKER_MAP = {
    "SPX": "^GSPC", 
    "XSP": "^XSP", 
    "DJI": "^DJI", 
    "IXIC": "^IXIC", 
    "BRK.B": "BRK-B"
}

# ==========================================
# 2. FUNCTIONS
# ==========================================

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50) 

def calculate_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def check_earnings_safe(ticker):
    """
    Checks if earnings are > 21 days away.
    """
    try:
        stock = yf.Ticker(ticker)
        earnings_dates = None
        
        try:
            cal = stock.calendar
            if cal is not None and not cal.empty:
                if isinstance(cal, pd.DataFrame):
                    if 'Earnings Date' in cal.index:
                        earnings_dates = cal.loc['Earnings Date']
                    elif 0 in cal.columns:
                        earnings_dates = cal[0]
                elif isinstance(cal, dict) and 'Earnings Date' in cal:
                    earnings_dates = cal['Earnings Date']
        except: pass
        
        if earnings_dates is not None:
            if isinstance(earnings_dates, list):
                next_date = earnings_dates[0]
            elif isinstance(earnings_dates, pd.Series):
                next_date = earnings_dates.iloc[0]
            else:
                next_date = earnings_dates

            next_date = pd.to_datetime(next_date).replace(tzinfo=None)
            today = datetime.now().replace(tzinfo=None)
            days = (next_date - today).days
            
            if days < 0: 
                return True, 90
            
            if days < EARNINGS_BUFFER:
                return False, days
            else:
                return True, days

        return True, 999 
    except Exception:
        return True, 999

def get_yahoo_daily(tickers):
    """
    Bulk downloads daily Close, High, and Low data from Yahoo Finance.
    """
    print(f"   [YAHOO] Downloading Daily data for {len(tickers)} symbols...")
    start_date = datetime.now() - timedelta(days=150)
    
    yahoo_tickers = []
    mapping = {} 
    
    for t in tickers:
        if t in TICKER_MAP:
            y_t = TICKER_MAP[t]
        else:
            y_t = t.replace(".", "-")
        yahoo_tickers.append(y_t)
        mapping[y_t] = t

    # Download all tickers at once
    df = yf.download(yahoo_tickers, start=start_date, progress=False)
    
    # Extract specific metric columns and rename headers back to original formats
    if isinstance(df.columns, pd.MultiIndex):
        df_close = df['Close'].rename(columns=mapping)
        df_high = df['High'].rename(columns=mapping)
        df_low = df['Low'].rename(columns=mapping)
    else:
        # Fallback if only 1 symbol somehow gets passed
        df_close = df[['Close']].rename(columns={'Close': mapping.get(yahoo_tickers[0], yahoo_tickers[0])})
        df_high = df[['High']].rename(columns={'High': mapping.get(yahoo_tickers[0], yahoo_tickers[0])})
        df_low = df[['Low']].rename(columns={'Low': mapping.get(yahoo_tickers[0], yahoo_tickers[0])})
            
    return df_close.sort_index(), df_high.sort_index(), df_low.sort_index()

def get_twelve_data_hourly(symbol):
    base_url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": symbol,
        "interval": "1h",
        "outputsize": 350, 
        "apikey": TD_API_KEY,
        "order": "ASC"
    }
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        if "values" in data:
            df = pd.DataFrame(data["values"])
            df['datetime'] = pd.to_datetime(df['datetime'])
            df['close'] = pd.to_numeric(df['close'])
            df.set_index('datetime', inplace=True)
            df.rename(columns={'close': 'Close'}, inplace=True)
            return df
        return pd.DataFrame()
    except:
        return pd.DataFrame()

# ==========================================
# 3. MAIN LOGIC
# ==========================================

def scan_market(ticker_list):
    # --- PHASE 1: MACRO (YAHOO) ---
    df_close, df_high, df_low = get_yahoo_daily(ticker_list)
    
    all_metrics = [] 
    filtered_out_count = 0
    print("\n   [PHASE 1] Calculating Deviations (All Stocks)...")
    
    for ticker in df_close.columns:
        try:
            closes = df_close[ticker].dropna()
            highs = df_high[ticker].dropna()
            lows = df_low[ticker].dropna()
            
            if len(closes) < LOOKBACK_DAYS: continue

            curr_close = float(closes.iloc[-1])
            
            # --- EDITED: PRICE RANGE FILTER ---
            # Filter out stocks less than MIN_PRICE or greater than MAX_PRICE
            if curr_close < MIN_PRICE or curr_close > MAX_PRICE:
                filtered_out_count += 1
                continue

            period_high = float(highs.tail(LOOKBACK_DAYS).max())
            period_low = float(lows.tail(LOOKBACK_DAYS).min())
            
            if period_high == period_low: continue
            
            location = (curr_close - period_low) / (period_high - period_low)
            deviation = abs(location - 0.5)
            
            # Pre-calculate type
            sig_type = 'None'
            if location <= DISCOUNT_LEVEL: sig_type = 'BUY'
            elif location >= PREMIUM_LEVEL: sig_type = 'SELL'
            
            if sig_type != 'None':
                all_metrics.append({
                    'ticker': ticker,
                    'location': location,
                    'deviation': deviation,
                    'strike': period_low if location < 0.5 else period_high,
                    'price': curr_close,
                    'type': sig_type
                })
                
        except Exception:
            continue
            
    # Sort by deviation to get the most extreme ones first
    all_metrics.sort(key=lambda x: x['deviation'], reverse=True)
    
    # --- EDITED: UPDATED PRINT STATEMENT ---
    print(f"   -> Filtered out {filtered_out_count} stocks outside the ${MIN_PRICE}-${MAX_PRICE} price range.")
    print(f"   -> Found {len(all_metrics)} valid candidates in Zone.")
    print(f"   -> Filtering Earnings & Scanning Top {TARGET_PROCESSED} Safe Stocks with Twelve Data...")
    
    # --- PHASE 2: SMART LOOP (FILTER EARNINGS -> THEN SCAN) ---
    
    opportunities = []
    processed_count = 0
    skipped_count = 0
    
    # Iterate through the ranked list
    for cand in all_metrics:
        # STOP CONDITION
        if processed_count >= TARGET_PROCESSED:
            break
            
        ticker = cand['ticker']
        
        # 1. CHECK EARNINGS (Fast)
        is_safe, days_to_earn = check_earnings_safe(ticker)
        
        if not is_safe:
            print(f"      [SKIP] {ticker} Earnings in {days_to_earn} days.")
            skipped_count += 1
            continue
            
        # 2. IF SAFE: PROCESS WITH TWELVE DATA
        processed_count += 1
        print(f"      Processing {processed_count}/{TARGET_PROCESSED}: {ticker} ({cand['type']})...", end=" ")
        
        try:
            # Fetch Data
            df_hourly = get_twelve_data_hourly(ticker)
            
            if df_hourly.empty:
                print("x No data.")
                time.sleep(8) 
                continue

            # Resample 1H -> 4H
            df_4h = df_hourly.resample('4h').agg({'Close': 'last'}).dropna()
            
            if len(df_4h) < 20: 
                print(f"x Too few candles.")
                time.sleep(8)
                continue
            
            # Calculate Indicators
            df_4h['EMA_TRIGGER'] = calculate_ema(df_4h['Close'], TRIGGER_EMA)
            df_4h['RSI'] = calculate_rsi(df_4h['Close'], RSI_PERIOD)
            
            curr = df_4h['Close'].iloc[-1]
            ema_trig = df_4h['EMA_TRIGGER'].iloc[-1]
            rsi = df_4h['RSI'].iloc[-1]
            
            sig = "None"
            status = "Waiting"
            
            if cand['type'] == 'BUY':
                if curr > ema_trig: 
                    if rsi < RSI_BUY_MAX:
                        sig = "BUY"
                        status = "CONFIRMED"
                    else:
                        status = "Uptrend (RSI High)" 
                else:
                    status = "Waiting (Price < EMA8)"

            elif cand['type'] == 'SELL':
                if curr < ema_trig:
                    if rsi > RSI_SELL_MIN:
                        sig = "SELL"
                        status = "CONFIRMED"
                    else:
                        status = "Downtrend (RSI Low)"
                else:
                    status = "Waiting (Price > EMA8)"
                
            if sig != "None":
                print(f"+ FOUND! (RSI: {round(rsi,1)})")
                opportunities.append({
                    'Ticker': ticker,
                    'Signal': sig,
                    'Price': round(curr, 2),
                    'Rec_Strike': round(cand['strike'], 2),
                    'RSI': round(rsi, 1),
                    'Link': f"https://finance.yahoo.com/quote/{ticker}"
                })
            else:
                print(f"- {status} (RSI: {round(rsi,1)})")
        
        except Exception as e:
            print(f"x Error: {e}")
        
        # RATE LIMIT SLEEP (TwelveData free tier limit)
        time.sleep(8)

    print(f"\n   [DONE] Processed {processed_count} stocks. Skipped {skipped_count} due to Earnings.")
    return pd.DataFrame(opportunities)

# ==========================================
# 4. OUTPUT
# ==========================================

if TD_API_KEY == "YOUR_TWELVE_DATA_API_KEY_HERE":
    print("ERROR: Please insert your Twelve Data Key at the top of the script.")
else:
    results = scan_market(tickers)

    print("\n" + "="*80)
    print(f" TIME FREEDOM SCANNER - {datetime.now().strftime('%Y-%m-%d %H:%M')} EST")
    print("="*80)

    if not results.empty:
        results = results.sort_values(by=['Signal'])
        print(results[['Ticker', 'Signal', 'Price', 'RSI', 'Rec_Strike', 'Link']].to_string(index=False))
        
        print("\n" + "-"*80)
        buys = results[results['Signal'] == "BUY"]['Ticker'].tolist()
        sells = results[results['Signal'] == "SELL"]['Ticker'].tolist()
        
        if buys: print(f"COPY BUYS: {','.join(buys)}")
        if sells: print(f"COPY SELLS: {','.join(sells)}")
    else:
        print("No confirmed 4H setups found.")